In [1]:
# Importing the Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, RobustScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
from kerastuner.tuners import Hyperband

# load the dataset
data = pd.read_excel('BBDM Project for 2nd research article.xlsx')

# Columns to exclude
columns_to_exclude = ['Chainage','Formation','RMC']

# Preprocessing: Exclude specified columns
X = data.drop(['PRnet'] + columns_to_exclude, axis=1)
Y = data['PRnet']

# Label encoding for multiple columns
label_encoder = LabelEncoder()
for col in ['Lithology', 'Weathering', 'Rock Strength']:
    X[col] = label_encoder.fit_transform(X[col])

# Train Test Split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# Data Standardization
scaler = RobustScaler()
scaler.fit(X_train)  # Fit on the training data
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Define a function that returns the model architecture based on the hyperparameters
def build_model(hp):
    model = Sequential()
    model.add(Dense(units=hp.Int('input_units', min_value=32, max_value=512, step=32),
                    input_dim=X_train.shape[1],  
                    activation=hp.Choice('input_activation', values=['relu', 'tanh', 'sigmoid']),
                    kernel_initializer=hp.Choice('input_kernel_initializer', values=['he_normal', 'glorot_uniform'])))
    
    for i in range(hp.Int('n_layers', 1, 5)):
        model.add(Dense(units=hp.Int(f'dense_{i}_units', min_value=32, max_value=512, step=32),
                        activation=hp.Choice(f'activation_{i}', values=['relu', 'tanh', 'sigmoid']),
                        kernel_initializer=hp.Choice(f'kernel_initializer_{i}', values=['he_normal', 'glorot_uniform'])))
    
    model.add(Dense(1, activation='linear'))
    
    # Compile the model
    model.compile(optimizer=hp.Choice('optimizer', values=['adam', 'sgd', 'rmsprop']),
                  loss='mean_squared_error',
                  metrics=['mean_squared_error'])
    
    return model

# Define Early Stopping Callback
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Setup the hyperparameter tuner with Hyperband
tuner = Hyperband(build_model,
                  max_epochs=100,
                  objective='val_mean_squared_error',
                  factor=3,
                  directory='model_tuning',
                  project_name='ANN_hyperparameter_tuning_v2',  # Change project name
                  overwrite=True)  # Use overwrite=True to avoid conflicts

# Start hyperparameter search with Hyperband
tuner.search(X_train_scaled, Y_train,
             validation_data=(X_test_scaled, Y_test),
             callbacks=[early_stopping],  # Include early stopping callback
             verbose=2)

# Get and evaluate the best model
best_model = tuner.get_best_models(num_models=1)[0]
loss, mean_squared_error = best_model.evaluate(X_test_scaled, Y_test)
print(f"Test Mean Squared Error: {mean_squared_error}")

# Print the best hyperparameters
best_hp = tuner.get_best_hyperparameters()[0]
print("Best Hyperparameters:")
print("Optimizer:", best_hp.get('optimizer'))
print("Kernel Initializer for Input Layer:", best_hp.get('input_kernel_initializer'))
print("Number of Hidden Layers:", best_hp.get('n_layers'))
print("Activation Function for Input Layer:", best_hp.get('input_activation'))
for i in range(best_hp.get('n_layers')):
    dense_layer_num = i + 1
    dense_units_param = f'dense_{i}_units'
    activation_param = f'activation_{i}'
    kernel_initializer_param = f'kernel_initializer_{i}'
    
    print(f"Number of Neurons in Dense Layer {dense_layer_num}:", best_hp.get(dense_units_param))
    print(f"Activation Function for Dense Layer {dense_layer_num}:", best_hp.get(activation_param))
    print(f"Kernel Initializer for Dense Layer {dense_layer_num}:", best_hp.get(kernel_initializer_param))

# Print the best epoch number
best_epoch = tuner.oracle.get_best_trials(1)[0].hyperparameters.values['tuner/epochs']
print("Best Epoch Number:", best_epoch)

Trial 206 Complete [00h 00m 11s]
val_mean_squared_error: 16.118724822998047

Best val_mean_squared_error So Far: 15.659162521362305
Total elapsed time: 00h 09m 09s
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 677us/step - loss: 16.9072 - mean_squared_error: 16.9072
Test Mean Squared Error: 15.659162521362305
Best Hyperparameters:
Optimizer: adam
Kernel Initializer for Input Layer: glorot_uniform
Number of Hidden Layers: 1
Activation Function for Input Layer: relu
Number of Neurons in Dense Layer 1: 352
Activation Function for Dense Layer 1: relu
Kernel Initializer for Dense Layer 1: he_normal
Best Epoch Number: 100
